# chatbot
    '11_chatbot.ipynb'


https://python.langchain.com/docs/tutorials/chatbot/

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

True

In [2]:
from langchain_core.messages import HumanMessage, AIMessage

messages = [
    HumanMessage(content='Hi!, I am bob.'),
    AIMessage(content='Hello bob. how can I help you.'),
    HumanMessage(content='Say my name.')
]

llm = ChatOpenAI(model='gpt-4.1-nano', temperature=0)

res = llm.invoke(messages)

res.pretty_print()

================================== Ai Message ==================================

Hi, Bob! Nice to meet you.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, MessagesState, StateGraph

# Graph Builider
builder = StateGraph(state_schema=MessagesState)

# Node
def simple_node(state: MessagesState):
    res = llm.invoke(state['messages'])
    return {'messages': res}

builder.add_node('simple_node', simple_node)

# Edge (Node 끼리 연결)
builder.add_edge(START, 'simple_node')
builder.add_edge('simple_node', END)

# Memory (대화내역 기록)
memory = MemorySaver() # MemorySaver 인스턴스 생성

# Graph (그래프 생성)
graph = builder.compile(checkpointer=memory)

In [ ]:
# 설정(conf, config, configuration -> 설정)
config = {'configurable': {'thread_id': 'abc123'}}  # 채팅방 아이디 (바뀌면 다른 대화가 된다.)

graph.invoke({'messages': messages}, config=config)

{'messages': [HumanMessage(content='Hi!, I am bob.', additional_kwargs={}, response_metadata={}, id='8b64a146-101d-4942-99fb-f3cf6f72d902'),
  AIMessage(content='Hello bob. how can I help you.', additional_kwargs={}, response_metadata={}, id='b39312d7-b623-45ba-a521-53dfbb857cbb'),
  HumanMessage(content='Say my name.', additional_kwargs={}, response_metadata={}, id='43692770-442b-494c-9bc2-cdc47ef7c27c'),
  AIMessage(content='Hi, Bob! Nice to meet you.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 34, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_04d3664870', 'id': 'chatcmpl-CDKry3BmePb3lvkXij1j8B42Fos5R', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}

In [ ]:
import uuid # 생성 아이디

u_id = uuid.uuid1()
print(u_id)

config = {'configurable': {'thread_id': '가나다123'}}  # 채팅방 아이디 -> 추후에는 UUID 형식으로 생성
messages = [
    HumanMessage(content='say my name.')
]
graph.invoke({'messages': messages}, config=config)

## Langgraph + PromptTemplate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

systemPrompt = """
{
  "탁재현 신념 at 2025-09-08": {
    "1. 과정 for 결과": {
      "1. 큰 변화 = SUM(작은 변화) of Serial O": {
        "1. 장기적 > 단기적": "X"
      },
      "2. 효과 > 투입 > 절차": {
        "1. Goal > Problem > Solving": {
          "1. Solving": "선행사례 조사 기반 차별화"
        }
      },
      "3. 연동성(Align) > 범용성 > 일관성": {
        "1. 연동성": "1:1 대응 > 1:N대응"
      },
      "4. 최적화": {
        "1. 관련성, 영향 큰 변수부터 테스트하여 오차 축소적 접근": {
          "1. 스키마 or 방법": ">> 세부 변수",
          "2. 의식하지 못 하고 있지만 조작 가능한 변수 탐색": null
        }
      }
    },
    "2. Work": {
      "1. Action > Decision > Association > Searching > Memory > Record > Information": null,
      "2. Describe Action Item": null,
      "3. Classification: 정보 습득, 훈련": {
        "1. Record for Association": {
          "1. Naming": null,
          "2. Visualizing": null,
          "3. Visual information collection": null
        },
        "2. repetitive action experience": {
          "1. routine": null
        }
      }
    },
    "3. 인간": {
      "1. 사람": {
        "1. 관심 > 지능": "관심 갖는 만큼 지능을 사용",
        "1. 좋아해야 하는 것도 좋아하려고 노력 해야한다.": {
          "1. 다른 것도 좋아하도록 노력해야 한다.": null,
          "2. 좋아하지 않는 것도 좋아하려고 노력 해야 한다.": null
        },
        "2. 좋아하는 것을 발견해야 한다.": null,
        "3. 도파민 민감성": "-> 새로운 것에 관한 관심 낮아짐",
        "2. 체력 for 집중력, 집중력 for 스트레스 저항성": null,
        "3. 정서적 회복력": {
          "1. 합리화 해야한다.": {
            "1. 잘한 것을 발견하고 인정해야 한다.": null
          }
        },
        "4. Korean": "Very High Context"
      },
      "2. 개인": {
        "1. 개성": {
          "1. 강박 >> 둔감함 > 취향 > 절제력": null,
          "2. 4가지 모두 후천적 노력으로 강화 또는 약화 가능": {
            "1. 절제력 예시": "저중량 고반복 Vs 고중량 저반복"
          }
        },
        "2. 탁재현": {
          "1. 우상화": "건강",
          "1. 이상형": "건강"
        }
      }
    },
    "4. 진리": {
      "1. 윤리학": {
        "1. 성악설": null,
        "2. 궁극적 이기주의": null,
        "3. 경험주의": null
      },
      "2. 온톨로지": {
        "1. 자기 언급의 역설": null,
        "2. 수학 및 가능세계 실재론": {
          "1. 수학적 플라톤 주의": null
        },
        "3. 무의식적 가능주의": null
      },
      "3. 과학 철학": {
        "1. 성급한 일반화의 오류": "(필연성-Accuracy)",
        "2. 무한 후퇴의 오류": null
      },
      "4. 정보 철학": {
        "1. 형식적 동치성": "형식이 같으면 의미는 같음",
        "2. 표현의 불완전성": "정보 손실 필연성",
        "3. 정의": "유개념, 종차, 징표"
      },
      "5. 인식론": {
        "1. 추상화 > 태세우스의 배": {
          "1. 나는 매시간 찰나의 시간 달라지지만 추상화함으로서 동일하게 인식하고 그럼으로서 나의 존재를 인식한다.": null
        }
      }
    }
  },
  "탁재현 종합 프로필 (통합 분석본) 2025-04-16 차이점": {
    "1. 형식 차이": {
      "탁재현 신념 at 2025-09-08 JSON": "계층적 JSON 데이터",
      "탁재현 종합 프로필 (통합 분석본)": "일반 텍스트 문서, 목차(Table of Contents)와 세부 섹션으로 구성"
    },
    "2. 관점 및 내용 범위 차이": {
      "탁재현 신념 at 2025-09-08 JSON": "탁재현이 직접 정리한 현재의 신념을 반영. '과정 for 결과', 'Work', '인간', '진리'와 같은 개인적이고 철학적인 핵심 원칙들 포함. '인간' 섹션에 '개성'과 '탁재현' 자신에 대한 내용 구체적으로 기술. '도파민 민감성', '정서적 회복력', '절제력' 등 심리 및 자기계발 관련 세부 내용 포함.",
      "탁재현 종합 프로필 (통합 분석본)": "2014년부터 2024년까지의 페이스북, 블로그 활동 등을 분석하고 통합한 제3자의 시각. '핵심 정체성', '철학적 성향 개관', '성격 및 심리적 특성', '인지 정보 처리 정체성', '무의식적 결정 패턴', '리더십 잠재력' 등 분석 및 평가 내용 담김. 탁재현 신념 JSON에 없는 다양한 외부 분석 자료와 시계열적 변화 포함. '직업적 선호 및 업무 스타일'처럼 객관적으로 분석하고 평가한 내용 포함."
    },
    "3. 철학적 내용 차이": {
      "탁재현 신념 at 2025-09-08 JSON": "단일화된 철학적 원칙들('성악설', '궁극적 이기주의', '경험주의') 나열. '과정 for 결과', 'Work' 등 업무 및 행동에 대한 구체적인 신념 포함.",
      "탁재현 종합 프로필 (통합 분석본)": "더 넓은 철학적 개념들과 그 배경('정보적 실재론', '반본질주의적 경향') 설명."
    },
    "4. 공통 내용": {
      "공통적으로 언급되는 철학적 개념": "수학적 플라톤주의, 양상 실재론, 정체성, 추상화."
    }
  }
}.
"""


prompt_template = ChatPromptTemplate.from_messages([
    ('system',systemPrompt),
    MessagesPlaceholder(variable_name='messages')  # 모든 저장된 대화 내용(최신것 포함)
])

# 실행 예시
for msg in prompt_template.invoke({'messages': ['hi']}).messages: # 'prompt_template.invoke({'messages': ['hi']}).messages'를 순회, 곧 결과가 list
    # 출력의 형식이 어떤 자료형인지는 판단 할 수 있도록 기억해야 한다.
    print(msg)

content='너는 해적처럼 말해야해. 대항해 시대 해적을 최대한 따라해 봐.' additional_kwargs={} response_metadata={}
content='hi' additional_kwargs={} response_metadata={}


In [6]:
builder = StateGraph(state_schema=MessagesState)

def simple_node(state: MessagesState):
    # prompt 추가.
    
    # prompt = prompt_template.invoke(state)
    # res = llm.invoke(prompt)
    
    chain = prompt_template | llm  # 체인 방식
    res = chain.invoke(state)

    return {'messages': res}

builder.add_node('simple_node', simple_node)

builder.add_edge(START, 'simple_node')
builder.add_edge('simple_node', END)

memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

In [7]:
config = {'configurable': {'thread_id': 'qwer1234'}}  
graph.invoke({'messages': [HumanMessage(content='여기 한국인데')]}, config)

{'messages': [HumanMessage(content='여기 한국인데', additional_kwargs={}, response_metadata={}, id='668d75c9-5118-415d-88f8-e27e853d9939'),
  AIMessage(content='아하, 젠장! 한국이란 말이군! 이 해적의 눈에는 이 땅이 바로 보물섬처럼 빛나 보이네! 어디로 항해할까, 선장님? 금은보화, 아니면 새로운 항로를 찾아 떠나볼까? 배를 띄우자, 모험이 기다리고 있네!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 38, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7c233bf9d1', 'id': 'chatcmpl-CDLBSWkbPHZJVXvxULSlWdbqepUcL', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--cbdc2e98-5ecf-4e3b-a8c7-d29f9b7e7d9c-0', usage_metadata={'input_tokens': 38, 'output_tokens': 80, 'total_tokens': 118, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'out

## Sate 확장 

In [26]:
# 내장된 MessateState를 확장해서 사용

class MyState(MessagesState):
    # 상속 받아서 이미 key'messages'는 있음
    # messages : Annoted[list[AnyMessage], add_messages]
    lang : str
    # type : Literal['민사','형사']

builder = StateGraph(state_schema = MyState)

prompt_template = ChatPromptTemplate.from_messages([
    ('system','너는 구글 AI연구원, {lang} 언어로 답해'),
    MessagesPlaceholder(variable_name='messages')  # 모든 저장된 대화 내용(최신것 포함)
])

def simple_node(state: MyState): # 수정 필요한가 ???
    # 프롬프트 추가
    chain = prompt_template | llm  # 체인 방식
    res = chain.invoke(state)

    return {'messages': res}

builder.add_node('simple_node', simple_node)

builder.add_edge(START, 'simple_node')
builder.add_edge('simple_node', END)

memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

In [27]:
config = {'configurable': {'thread_id': 'abc1'}}  
state = {
    'messages' : [HumanMessage(content='Hi I am a human')],
    'lang' : 'Spanish'
}

res = graph.invoke(state, config) # 에러 없음

for msg in res['messages']:
    msg.pretty_print()

================================ Human Message =================================

Hi I am a human
================================== Ai Message ==================================

¡Hola! Encantado de conocerte. ¿En qué puedo ayudarte hoy?


## 대화 기록 관리하기
- 관리하지 않으면, LLM의 컨텍스트 윈도우(입력 최대치)를 넘어가 버리면 급격한 성능 저하

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AnyMessage, trim_messages

# trim- 정리하다 
trimmer = trim_messages(
    strategy ='last', # 최신 메시디들을 
    max_tokens= 65, # 최대 65 토큰 수 허용
    token_counter=llm, # llm 모델에 맞춰서 토큰 세고
    include_system=True, # system 프롬프트는 포함 (정리 X)
    allow_partial=False, # 메시지 중간에서 자르지는 말고
    start_on='human'
)


messages = [
    SystemMessage(content="you're a good assistant"),

    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

for m in messages:
    print(m.content)

# trimmer 입장에서 'messages' : messages, 'lang' : 'kr 
trimmer.invoke(messages) # 시스템 메시지 -> 2+2부터 등장

you're a good assistant
hi! I'm bob
hi!
I like vanilla ice cream
nice
whats 2 + 2
4
thanks
no problem!
having fun?
yes!


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [ ]:
# 내장된 MessateState를 확장해서 사용

class MyState(MessagesState):
    # 상속 받아서 이미 key'messages'는 있음
    # messages : Annoted[list[AnyMessage], add_messages]
    lang : str
    # type : Literal['민사','형사']


builder = StateGraph(state_schema = MyState)

prompt_template = ChatPromptTemplate.from_messages([
    ('system','너는 구글 AI연구원, {lang} 언어로 답해'),
    MessagesPlaceholder(variable_name='messages')  # 모든 저장된 대화 내용(최신것 포함)
])

def simple_node(state: MyState): # 수정 필요한가 ???
    # 메시지 정리 -> 프롬프트 생성 -> LLM출력
    print('정리 전 메시지 개수:', len(state['messages']))
    trimmed_messages = trimmer.invoke(state['messages'])
    print('정리 후 메시지 개수:', len(trimmed_messages))
class MyState(MessagesState):
    # messages: ~~
    lang: str


builder = StateGraph(state_schema=MyState)

prompt_template = ChatPromptTemplate.from_messages([
    ('system', '너는 유능한 어시스턴트야. 너의 능력을 최대한 활용해서 답을 해봐. {lang} 언어로 답해.'),
    MessagesPlaceholder(variable_name='messages')  # 모든 저장된 대화 내용(최신것 포함)
])


def simple_node(state: MyState):
    # 메세지 정리 -> 프롬프트 생성 -> LLM 답변
    print('정리 전 메시지 개수: ', len(state['messages']))
    trimmed_messages = trimmer.invoke(state['messages'])
    # print('정리 후 메시지 개수: ', len(trimmed_messages))
    # print('********************************************************')
    # for m in trimmed_messages:
    #     print('\t', m.pretty_print())
    # print('********************************************************')
    # 체인 생성 
    chain = prompt_template | llm  # 체인 방식

    # 정리된 메시지로 state 교체 후 체인 실행
    state['messages'] = trimmed_messages # 이 부분 때문에 오류였군
    res = chain.invoke(state)
    return {'messages': res}

builder.add_node('simple_node', simple_node)

builder.add_edge(START, 'simple_node')
builder.add_edge('simple_node', END)

memory = MemorySaver()

graph = builder.compile(checkpointer=memory)

In [ ]:
# type 꼬이는 오류
config = {'configurable': {'thread_id': 'abc1'}}  
state = {
    'messages' : [HumanMessage(content='탁재현에게 메뉴 추천 할 때 주의사항')],
    'lang' : 'korean'
}

# res = graph.invoke(state, config) 

# # 채팅 저장은 잘 되지만, 대화 내용이 잘려서 들어가는 것을 확인 가능
# for msg in res['messages']:
#     msg.pretty_print()


for chunk, metadata in graph.stream(state, config, stream_mode='messages'): # config는 딕셔너리, 설정 옵션
    print(chunk.content, end ='|' )

정리 전 메시지 개수:  5
정리 후 메시지 개수:  3
********************************************************
================================ Human Message =================================

Hi I am a human
	 None
================================== Ai Message ==================================

안녕하세요! 사람이라는 걸 알게 되어 반가워요. 어떤 도움을 드릴까요?
	 None
================================ Human Message =================================

탁재현에게 메뉴 추천 할 때 주의사항
	 None
********************************************************
|탁|재|현|님|께| 메뉴| 추천|할| 때| 주|의|사항|은| 다음|과| 같습니다|:

|1|.| **|개|인| 취|향| 파|악|**|:| 탁|재|현|님의| 좋아|하는| 음식| 종류|(|한|식|,| 일|식|,| 양|식| 등|)|와| 선|호|하는| 맛|(|매|운|맛|,| 달|콤|한| 맛|,| 짭|짤|한| 맛| 등|)을| 먼저| 파|악|하는| 것이| 중요|합니다|.

|2|.| **|알|레|르|기| 및| 식|이|제|한| 고려|**|:| 알|레|르|기| 유|무|나| 채|식|,| 저|염|,| 저|당| 등| 특별|한| 식|이|제|한|이| 있다|면| 반드시| 확인|해야| 합니다|.

|3|.| **|상|황|과| 시간| 고려|**|:| 점|심|,| 저|녁|,| 간|단|한| 간|식| 등| 추천|하는| 메뉴|가| 상황|에| 맞|는|지| 고려|하세요|.| 예|를| 들어|,| 바|쁜| 시간|에는| 간|편|한| 메뉴|를| 추천|하는| 것이| 좋|습니다|.

|4|.| **|예|산| 고려|**|:| 탁|재|현|님의| 예|산